# GenAI-Ready Enterprise Data Warehouse (CSN447, Group 8)
Run the cells **top to bottom**. Every output below is screenshot-ready for the report.

**Before you start:** in Colab, run Cell 1 and upload `genai-enterprise-data-warehouse.zip` when the upload button appears.

## 1. Upload the project and set up

In [ ]:
import os, sys, zipfile
try:
    from google.colab import files
    print("Choose genai-enterprise-data-warehouse.zip when the upload button appears")
    uploaded = files.upload()
    zipfile.ZipFile(next(iter(uploaded))).extractall(".")
    os.chdir("genai-enterprise-data-warehouse")
except ImportError:
    print("Not running in Colab - using the current folder.")
sys.path.insert(0, os.getcwd())
print("Working folder:", os.getcwd())

In [ ]:
!pip install -q anthropic python-dotenv requests

## 2. Build the data warehouse (ETL) and run the segmentation
Shows the validation check, table sizes and the K-Means segment summary.

In [ ]:
!python run_pipeline.py

## 3. OLAP operations

In [ ]:
import pandas as pd, plotly.express as px
import olap_mining.olap_queries as olap
pd.set_option("display.width", 140)

# ROLL-UP: aggregate up the location hierarchy (city > state > region > country)
data = olap.rollup_location("region")
display(data)
px.bar(data, x="region", y="total_sales", title="Roll-up: total sales by region").show()

In [ ]:
# DRILL-DOWN: go down the time hierarchy (year > quarter > month)
data = olap.drilldown_time("quarter")
data["period"] = data["year"].astype(str) + "-Q" + data["quarter"].astype(str)
display(data)
px.line(data, x="period", y="total_sales", markers=True, title="Drill-down: sales by quarter").show()

In [ ]:
# SLICE: fix ONE dimension (category = Technology)
data = olap.slice_cube("category", "Technology", ["year", "region"])
display(data)
px.bar(data, x="year", y="total_sales", color="region", barmode="group",
       title="Slice: Technology sales by year and region").show()

In [ ]:
# DICE: fix SEVERAL dimensions (Technology + Furniture, West + East, 2017-2018)
data = olap.dice_cube({"category": ["Technology", "Furniture"], "region": ["West", "East"],
                       "year": [2017, 2018]}, ["category", "region"])
display(data)
px.bar(data, x="region", y="total_sales", color="category", barmode="group",
       title="Dice: sub-cube of the sales cube").show()

In [ ]:
# PIVOT: rotate the cube (regions as rows, years as columns, with totals)
display(olap.pivot("region", "year").style.format("{:,.0f}"))

## 4. Data mining: customer segmentation (RFM + K-Means)

In [ ]:
from olap_mining.customer_segmentation import evaluate_k, run_segmentation, segment_summary

scores = evaluate_k()
display(scores.round(3))
px.line(scores, x="k", y="inertia", markers=True, title="Elbow curve (inertia vs k)").show()

In [ ]:
rfm = run_segmentation(k=4, save=True)
display(segment_summary(rfm))
px.scatter(rfm, x="frequency", y="monetary", color="rfm_segment", hover_name="customer_name",
           log_y=True, opacity=0.75, title="Customers by RFM segment (k=4)",
           labels={"frequency": "Orders (frequency)", "monetary": "Total sales, $ (log scale)"}).show()

## 5. GenAI layer: ask the warehouse in plain English
Paste an API key when asked: an **Anthropic** key (`sk-ant-...`) or a **Groq** key (`gsk_...`). You can give a second key as a backup.
Leave both empty to use the offline backup queries. The notebook never fails: if a key has no credits or hits a rate limit, it switches to the other key, then to backup queries.

In [ ]:
import getpass
for label in ["Paste an API key (sk-ant-... or gsk_...), or press Enter to skip: ",
              "Optional second key as a backup, or press Enter to skip: "]:
    try:
        key = getpass.getpass(label).strip()
    except Exception:
        key = ""
    if key.startswith("gsk_"):
        os.environ["GROQ_API_KEY"] = key
    elif key:
        os.environ["ANTHROPIC_API_KEY"] = key

from genai.text_to_sql import ask, get_providers
print("AI providers in use:", [(p.name, p.model) for p in get_providers()] or "none (backup mode)")

def show(question):
    r = ask(question)
    print("QUESTION:", question)
    who = r["source"] + (f" ({r['model']})" if r["model"] else "") + (" - from cache" if r["cached"] else "")
    print("ANSWERED BY:", who)
    if r["note"]:
        print("NOTE:", r["note"])
    if r["error"]:
        print("ERROR:", r["error"])
    if r["summary"]:
        print("ANSWER:", r["summary"])
    if r["sql"]:
        print("\nGENERATED SQL:\n" + r["sql"])
    if r["df"] is not None:
        display(r["df"])
    print("\n" + "-" * 90 + "\n")

In [ ]:
show("Which region has the highest total sales?")
show("Top 5 products by sales in 2017")
show("Monthly sales trend for 2018")

In [ ]:
# Run the SAME questions again: answers now come from cache (no API call, no credits used)
show("Which region has the highest total sales?")

In [ ]:
# Tricky questions: no profit column in the data, and the assistant is read-only (works even without a key)
show("What is the total profit?")
show("Delete all rows from the sales table")

## 6. Safety tests (screenshot this output for the report)

In [ ]:
!python -m tests.test_genai

## 7. Launch the Streamlit app (public link)
Run this AFTER Section 5 (so your API key is set). It starts the app and prints a public `https://....trycloudflare.com` link. Open it in a new tab.
The link works only while this cell keeps running. Stop the cell (square button) when you are done, because anyone with the link can use your API credits.

In [ ]:
import os, re, subprocess, time, urllib.request

!pip install -q streamlit

# 1. start the Streamlit app in the background (it inherits the API keys set in Section 5)
app = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
    stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)

ready = False
for _ in range(90):                                   # wait up to ~90 s
    try:
        ready = urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=2).read() == b"ok"
        if ready:
            break
    except Exception:
        time.sleep(1)
print("Streamlit server ready:", ready)

# 2. public tunnel with cloudflared (free, no account needed)
if not os.path.exists("cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "cloudflared")
    os.chmod("cloudflared", 0o755)
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                          stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)

url = None
for _ in range(60):
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        url = m.group(0)
        break
    time.sleep(1)

if url:
    print("\nOPEN THIS LINK:", url)
    print("(keep this cell running; first load can take ~15 s while the warehouse is built)")
else:
    print("No link was created. Log:\n", open("tunnel.log").read()[-1500:])